# Speech-to-Reasoning Pipeline: Whisper + Quantized Reasoning LLM

**Speech recognition:** OpenAI Whisper (`openai/whisper-small`, fp16)
**Reasoning model:** `unsloth/DeepSeek-R1-Distill-Qwen-7B-unsloth-bnb-4bit` (Unsloth dynamic 4-bit)
**Flow:** audio file -> 16 kHz mono wav -> Whisper -> text -> reasoning model -> thinking + final answer

Before running: `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`.

## Step 1: GPU and ffmpeg check

In [1]:
!nvidia-smi
!ffmpeg -version | head -n 1
import torch
print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0))

Sat Nov  7 17:09:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 2: Install libraries

In [2]:
%%capture
!pip install unsloth
!pip install -q gTTS soundfile

## Step 3: Create sample audio queries
Five spoken questions are created with Google text-to-speech so the notebook works without any recording. Four are short. The fifth is a long problem of about 100 words, written to be longer than 30 seconds so that it tests the audio chunking (the printed durations in Step 4 show whether it was). Each question has a known answer so the final answers can be checked. You can use your own voice recording in Step 12.

In [3]:
import os, time
from gtts import gTTS
from IPython.display import Audio, display

os.makedirs("audio", exist_ok=True)

# name: (spoken text, expected answer)
QUERIES = {
    "q1_train": ("If a train travels at sixty kilometers per hour for two and a half hours, how far does it go?", "150"),
    "q2_height": ("Alice is taller than Bob, and Bob is taller than Carol. Who is the shortest of the three?", "carol"),
    "q3_discount": ("A shirt costs twenty dollars after a twenty percent discount. What was the original price?", "25"),
    "q4_tablets": ("A patient takes two tablets every six hours. How many tablets does the patient take in two days?", "16"),
    "q5_pharmacy": ("Please listen carefully and solve this problem step by step. A pharmacy sells vitamin tablets in two kinds of bottles. "
                    "A small bottle holds thirty tablets and costs six dollars. A large bottle holds ninety tablets and costs fifteen dollars. "
                    "A customer needs at least two hundred tablets and wants to spend as little money as possible. She can buy any number of "
                    "small bottles and any number of large bottles. Work out how many bottles of each kind she should buy, and tell me the "
                    "total price she will pay at the end.", "36"),
}

for name, (text, _) in QUERIES.items():
    gTTS(text=text, lang="en").save(f"audio/{name}.mp3")

for f in sorted(os.listdir("audio")):
    print(f"audio/{f:<20} {os.path.getsize('audio/' + f) / 1024:6.1f} KB")
display(Audio("audio/q1_train.mp3"))

audio/q1_train.mp3           52.1 KB
audio/q2_height.mp3          52.1 KB
audio/q3_discount.mp3        51.6 KB
audio/q4_tablets.mp3         59.4 KB
audio/q5_pharmacy.mp3       282.9 KB


## Step 4: Audio loading and encoding
Audio can come in any format (mp3, m4a, wav...). Whisper needs 16 kHz mono samples, so every file is converted with ffmpeg first and then read as a float32 array.

In [4]:
import subprocess
import numpy as np
import soundfile as sf

SR = 16000   # Whisper sampling rate

def load_audio(path):
    wav_path = path.rsplit(".", 1)[0] + "_16k.wav"
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", path, "-ar", str(SR), "-ac", "1", wav_path], check=True)
    data, sr = sf.read(wav_path, dtype="float32")
    return data

audio_paths = [f"audio/{k}.mp3" for k in QUERIES]
waves = [load_audio(p) for p in audio_paths]
for p, w in zip(audio_paths, waves):
    print(f"{p:<26} {len(w) / SR:5.1f} sec | {len(w):>8,} samples | {SR} Hz mono")

audio/q1_train.mp3           6.7 sec |  106,752 samples | 16000 Hz mono
audio/q2_height.mp3          6.7 sec |  106,752 samples | 16000 Hz mono
audio/q3_discount.mp3        6.6 sec |  105,600 samples | 16000 Hz mono
audio/q4_tablets.mp3         7.6 sec |  121,728 samples | 16000 Hz mono
audio/q5_pharmacy.mp3       36.2 sec |  579,456 samples | 16000 Hz mono


## Step 5: Load Whisper
Whisper runs in fp16 on the GPU. The helper `gpu_mem()` is used to record the GPU memory at every stage.

In [5]:
from unsloth import FastLanguageModel   # unsloth is imported before transformers
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import gc

def gpu_gb(peak=False):
    f = torch.cuda.max_memory_allocated if peak else torch.cuda.memory_allocated
    return f() / 1024**3

mem = {}
W = {}

def load_whisper(name="openai/whisper-small"):
    W["proc"] = WhisperProcessor.from_pretrained(name)
    try:
        m = WhisperForConditionalGeneration.from_pretrained(name, dtype=torch.float16)
    except TypeError:
        m = WhisperForConditionalGeneration.from_pretrained(name, torch_dtype=torch.float16)
    W["model"] = m.to("cuda").eval()

def free_whisper():
    W.clear()
    gc.collect()
    torch.cuda.empty_cache()

load_whisper()
mem["whisper_loaded"] = gpu_gb()
n_params = sum(p.numel() for p in W["model"].parameters()) / 1e6
print(f"Whisper loaded: {n_params:.0f}M parameters (fp16)")
print(f"GPU memory allocated: {mem['whisper_loaded']:.2f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1107 17:11:31.960000 1102 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1107 17:11:32.023000 1102 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!


preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

... (7 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

Whisper loaded: 242M parameters (fp16)
GPU memory allocated: 0.48 GB


## Step 6: Transcribe the audio (chunking and batching)
Whisper reads 30 seconds of audio at a time, so longer files are cut into 30 second chunks. All chunks from all files are processed in batches, and the texts are joined back for each file.

In [6]:
def chunk_audio(wave, chunk_s=30):
    n = chunk_s * SR
    return [wave[i:i + n] for i in range(0, len(wave), n)]

def transcribe(waves, batch_size=8, language="en"):
    chunks, owner = [], []
    for idx, w in enumerate(waves):
        for c in chunk_audio(w):
            chunks.append(c)
            owner.append(idx)
    texts = []
    for i in range(0, len(chunks), batch_size):
        batch = chunks[i:i + batch_size]
        feats = W["proc"](batch, sampling_rate=SR, return_tensors="pt").input_features.to("cuda", dtype=torch.float16)
        with torch.inference_mode():
            ids = W["model"].generate(feats, language=language, task="transcribe", max_new_tokens=224)
        texts += W["proc"].batch_decode(ids, skip_special_tokens=True)
    out = [""] * len(waves)
    for idx, t in zip(owner, texts):
        out[idx] = (out[idx] + " " + t.strip()).strip()
    return out, len(chunks)

torch.cuda.reset_peak_memory_stats()
t0 = time.time()
transcripts, n_chunks = transcribe(waves, batch_size=8)
t_asr = time.time() - t0
mem["whisper_peak"] = gpu_gb(peak=True)

print(f"{len(waves)} audio files -> {n_chunks} chunks -> transcribed in {t_asr:.1f} sec | peak GPU memory {mem['whisper_peak']:.2f} GB\n")
for p, t in zip(audio_paths, transcripts):
    print(f"[{p}]\n  {t}\n")

Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=224) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuati

5 audio files -> 6 chunks -> transcribed in 16.3 sec | peak GPU memory 0.84 GB

[audio/q1_train.mp3]
  If a train travels at 60 km per hour for 2.5 hours, how far does it go?

[audio/q2_height.mp3]
  Alice is taller than Bob, and Bob is taller than Carol. Who is the shortest of the three?

[audio/q3_discount.mp3]
  A shirt costs $20 after a 20% discount. What was the original price?

[audio/q4_tablets.mp3]
  A patient takes two tablets every six hours. How many tablets does the patient take in two days?

[audio/q5_pharmacy.mp3]
  Please listen carefully and solve this problem step by step. A pharmacy sells vitamin tablets in two kinds of bottles. A small bottle holds 30 tablets and costs $6. A large bottle holds 90 tablets and costs $15. A customer needs at least 200 tablets and wants to spend as little money as possible. She can buy any number of small bottles and any number of large bottles. any bottles of each kind she should buy and tell me the total price she will pay at the end.


### Transcription quality (word error rate)
The transcripts are compared with the text that was spoken. Word error rate (WER) is the share of words that were substituted, missed or added. The transcripts are also saved as UTF-8 text.

In [7]:
import re, json

def norm(t):
    return re.sub(r"[^a-z0-9' ]", " ", t.lower()).split()

def wer(ref, hyp):
    r, h = norm(ref), norm(hyp)
    d = list(range(len(h) + 1))
    for i in range(1, len(r) + 1):
        prev = d[:]
        d[0] = i
        for j in range(1, len(h) + 1):
            d[j] = min(prev[j] + 1, d[j - 1] + 1, prev[j - 1] + (r[i - 1] != h[j - 1]))
    return d[len(h)] / max(1, len(r))

wers = [wer(QUERIES[k][0], t) for k, t in zip(QUERIES, transcripts)]
for k, w in zip(QUERIES, wers):
    print(f"{k:<14} WER = {w * 100:5.1f}%")
print(f"\nAverage WER: {sum(wers) / len(wers) * 100:.1f}%")

with open("transcripts.json", "w", encoding="utf-8") as f:
    json.dump(dict(zip(QUERIES, transcripts)), f, ensure_ascii=False, indent=2)

q1_train       WER =  30.0%
q2_height      WER =   0.0%
q3_discount    WER =  26.7%
q4_tablets     WER =   0.0%
q5_pharmacy    WER =  12.8%

Average WER: 13.9%


## Step 7: Free the Whisper memory
Whisper is not needed while the reasoning model is working, so it is removed from the GPU before the large model is loaded. This keeps the peak memory lower.

In [8]:
free_whisper()
mem["after_free"] = gpu_gb()
print(f"GPU memory allocated after freeing Whisper: {mem['after_free']:.2f} GB")

GPU memory allocated after freeing Whisper: 0.01 GB


## Step 8: Load the reasoning model (Unsloth dynamic 4-bit)
DeepSeek-R1-Distill-Qwen-7B is a Qwen model that was fine-tuned on reasoning data from DeepSeek-R1. It writes its thinking between `<think>` and `</think>` and then gives the final answer. The `-unsloth-bnb-4bit` version is the dynamic 4-bit quantization.

In [9]:
MODEL_NAME = "unsloth/DeepSeek-R1-Distill-Qwen-7B-unsloth-bnb-4bit"   # smaller option: unsloth/DeepSeek-R1-Distill-Qwen-1.5B-unsloth-bnb-4bit
max_seq_length = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)
tokenizer.padding_side = "left"          # needed for batched generation
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

mem["llm_loaded"] = gpu_gb()
total = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"Reasoning model loaded | GPU memory allocated: {mem['llm_loaded']:.2f} GB of {total:.2f} GB")

==((====))==  Unsloth 2026.11.3: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/100k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Unsloth: Will load unsloth/DeepSeek-R1-Distill-Qwen-7B-unsloth-bnb-4bit as a legacy tokenizer.


Reasoning model loaded | GPU memory allocated: 7.91 GB of 14.56 GB


### Which layers are 4-bit and which are kept in higher precision?

In [10]:
import bitsandbytes as bnb
import torch.nn as nn

q4 = [n for n, m in model.named_modules() if isinstance(m, bnb.nn.Linear4bit)]
full = [n for n, m in model.named_modules() if isinstance(m, nn.Linear) and not isinstance(m, bnb.nn.Linear4bit)]
print("Linear layers in 4-bit      :", len(q4))
print("Linear layers NOT quantized :", len(full))
for n in full[:10]:
    print("  -", n)

Linear layers in 4-bit      : 149
Linear layers NOT quantized : 48
  - model.layers.0.self_attn.q_proj
  - model.layers.0.self_attn.k_proj
  - model.layers.0.self_attn.v_proj
  - model.layers.0.self_attn.o_proj
  - model.layers.0.mlp.gate_proj
  - model.layers.0.mlp.up_proj
  - model.layers.0.mlp.down_proj
  - model.layers.1.self_attn.q_proj
  - model.layers.1.self_attn.k_proj
  - model.layers.1.self_attn.v_proj


## Step 9: Pass the transcripts to the reasoning model
The transcribed text is sent as the user message. All questions are generated together in one batch (with left padding). The output is split into the thinking part and the final answer.

In [11]:
INSTRUCTION = "Think step by step, then give the final answer in one short sentence."

def split_reasoning(text):
    if "</think>" in text:
        think, ans = text.split("</think>", 1)
        return think.replace("<think>", "").strip(), ans.strip(), True
    return text.replace("<think>", "").strip(), "", False     # thinking was cut off by the token limit

def reason(questions, max_new_tokens=1200, batch_size=5):
    FastLanguageModel.for_inference(model)
    results = []
    for i in range(0, len(questions), batch_size):
        qs = questions[i:i + batch_size]
        prompts = [tokenizer.apply_chat_template([{"role": "user", "content": f"{q}\n\n{INSTRUCTION}"}],
                                                 tokenize=False, add_generation_prompt=True) for q in qs]
        enc = tokenizer(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        L = enc["input_ids"].shape[1]
        t0 = time.time()
        with torch.inference_mode():
            out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.6,
                                 top_p=0.95, use_cache=True, pad_token_id=tokenizer.pad_token_id)
        dt = time.time() - t0
        for j, q in enumerate(qs):
            gen = out[j][L:]
            n_tok = int((gen != tokenizer.pad_token_id).sum())
            think, ans, done = split_reasoning(tokenizer.decode(gen, skip_special_tokens=True))
            results.append(dict(question=q, thinking=think, answer=ans, finished=done, tokens=n_tok, seconds=dt))
    return results

In [12]:
torch.manual_seed(0)
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
results = reason(transcripts, max_new_tokens=1200, batch_size=5)
t_llm = time.time() - t0
mem["llm_peak"] = gpu_gb(peak=True)

for k, r in zip(QUERIES, results):
    print("=" * 90)
    print(f"[{k}] Question (from audio): {r['question']}")
    shown = r["thinking"][:450] + (" ..." if len(r["thinking"]) > 450 else "")
    print(f"\nThinking ({r['tokens']} tokens):\n{shown}")
    print(f"\nFinal answer: {r['answer'] if r['finished'] else '(thinking was cut off before the answer)'}")
print("=" * 90)
print(f"\nReasoning time for the batch: {t_llm:.1f} sec | peak GPU memory {mem['llm_peak']:.2f} GB")

Both `max_new_tokens` (=1200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[q1_train] Question (from audio): If a train travels at 60 km per hour for 2.5 hours, how far does it go?

Thinking (216 tokens):
First, I need to determine how far the train travels. I know that distance is calculated by multiplying speed by time. The train's speed is 60 km/h, and it travels for 2.5 hours. 

Next, I'll perform the multiplication: 60 km/h multiplied by 2.5 hours. 

Calculating this gives 150 km. 

Therefore, the train travels a distance of 150 kilometers.

Final answer: To find the distance the train travels, use the formula:

\[
\text{Distance} = \text{Speed} \times \text{Time}
\]

Given:
- Speed = 60 km/h
- Time = 2.5 hours

Calculate the distance:

\[
\text{Distance} = 60 \, \text{km/h} \times 2.5 \, \text{h} = 150 \, \text{km}
\]

\[
\boxed{150\,\text{km}}
\]
[q2_height] Question (from audio): Alice is taller than Bob, and Bob is taller than Carol. Who is the shortest of the three?

Thinking (267 tokens):
Okay, so I need to figure out who is the shortest among Alic

## Step 10: Check the answers

In [13]:
import pandas as pd

rows = []
for (k, (_, expected)), w, r in zip(QUERIES.items(), wers, results):
    ok = r["finished"] and expected.lower() in r["answer"].lower()
    rows.append({"Query": k, "WER %": round(w * 100, 1), "Expected": expected,
                 "Final answer": r["answer"][:70], "Thinking finished": "yes" if r["finished"] else "no",
                 "Correct": "yes" if ok else "no", "Tokens": r["tokens"]})
df = pd.DataFrame(rows)
print(f"Correct answers: {(df['Correct'] == 'yes').sum()} of {len(df)}")
df

Correct answers: 4 of 5


,Query,WER %,Expected,Final answer,Thinking finished,Correct,Tokens
0,q1_train,30.0,150,"To find the distance the train travels, use th...",yes,yes,216
1,q2_height,0.0,carol,Carol is the shortest of the three.\n\nCarol i...,yes,yes,267
2,q3_discount,26.7,25,To determine the original price of the shirt b...,yes,yes,286
3,q4_tablets,0.0,16,**Solution:**\n\n1. **Calculate the number of ...,yes,yes,285
4,q5_pharmacy,12.8,36,**Solution:**\n\nTo determine the most cost-ef...,yes,no,544


## Step 11: Full pipeline for one audio file
One function runs everything for a single audio file: convert, transcribe with Whisper (loaded only for this step and freed again), and reason with the model that is already loaded.

In [14]:
def speech_to_reasoning(audio_path, max_new_tokens=1200):
    t0 = time.time()
    wave = load_audio(audio_path)
    load_whisper()
    text, _ = transcribe([wave])
    free_whisper()
    t_asr = time.time() - t0

    t1 = time.time()
    r = reason([text[0]], max_new_tokens=max_new_tokens)[0]
    t_llm = time.time() - t1

    print(f"Audio file : {audio_path} ({len(wave) / SR:.1f} sec)")
    print(f"Transcript : {text[0]}")
    print(f"\nThinking   : {r['thinking'][:400]}{' ...' if len(r['thinking']) > 400 else ''}")
    print(f"\nAnswer     : {r['answer'] if r['finished'] else '(thinking was cut off before the answer)'}")
    print(f"\nWhisper stage {t_asr:.1f} sec | reasoning stage {t_llm:.1f} sec ({r['tokens']} tokens) | peak GPU {gpu_gb(peak=True):.2f} GB")
    return text[0], r

torch.manual_seed(0)
_ = speech_to_reasoning("audio/q2_height.mp3")

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Both `max_new_tokens` (=224) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Audio file : audio/q2_height.mp3 (6.7 sec)
Transcript : Alice is taller than Bob, and Bob is taller than Carol. Who is the shortest of the three?

Thinking   : First, I note that Alice is taller than Bob, which means Alice > Bob.

Next, I observe that Bob is taller than Carol, so Bob > Carol.

Combining these two pieces of information, I can deduce that Alice > Bob > Carol.

Therefore, Carol is the shortest of the three.

Answer     : **Solution:**

1. **Given Information:**
   - Alice is taller than Bob: \( \text{Alice} > \text{Bob} \)
   - Bob is taller than Carol: \( \text{Bob} > \text{Carol} \)

2. **Combine the Information:**
   - From the first statement: \( \text{Alice} > \text{Bob} \)
   - From the second statement: \( \text{Bob} > \text{Carol} \)
   - Combining these: \( \text{Alice} > \text{Bob} > \text{Carol} \)

3. **Conclusion:**
   - Carol is the shortest.

**Final Answer:**

\boxed{\text{Carol}}

Whisper stage 11.8 sec | reasoning stage 12.7 sec (221 tokens) | peak GPU 8

## Step 12: Try your own voice
Record a question on your phone (any audio format works), set `USE_OWN_AUDIO = True`, run the cell and upload the file.

In [15]:
USE_OWN_AUDIO = False

if USE_OWN_AUDIO:
    from google.colab import files
    uploaded = files.upload()
    own_path = list(uploaded.keys())[0]
    _ = speech_to_reasoning(own_path)

## Step 13: Memory and time summary

In [16]:
total = torch.cuda.get_device_properties(0).total_memory / 1024**3
rows = [
    ("Whisper-small loaded (fp16)", mem["whisper_loaded"]),
    (f"Peak during transcription ({len(waves)} files, {n_chunks} chunks)", mem["whisper_peak"]),
    ("After freeing Whisper", mem["after_free"]),
    ("Reasoning model loaded (dynamic 4-bit)", mem["llm_loaded"]),
    (f"Peak during reasoning (batch of {len(results)})", mem["llm_peak"]),
]
print(f"{'Stage':<50}{'GPU memory':>12}")
print("-" * 62)
for name, v in rows:
    print(f"{name:<50}{v:>9.2f} GB")
print("-" * 62)
print(f"GPU total: {total:.2f} GB | highest peak: {max(v for _, v in rows):.2f} GB ({max(v for _, v in rows) / total * 100:.1f}%)")
print(f"\nWhisper time: {t_asr:.1f} sec | reasoning time: {t_llm:.1f} sec | generated tokens: {sum(r['tokens'] for r in results)}")

Stage                                               GPU memory
--------------------------------------------------------------
Whisper-small loaded (fp16)                            0.48 GB
Peak during transcription (5 files, 6 chunks)          0.84 GB
After freeing Whisper                                  0.01 GB
Reasoning model loaded (dynamic 4-bit)                 7.91 GB
Peak during reasoning (batch of 5)                     8.28 GB
--------------------------------------------------------------
GPU total: 14.56 GB | highest peak: 8.28 GB (56.8%)

Whisper time: 16.3 sec | reasoning time: 48.2 sec | generated tokens: 1598


## Summary
- Whisper-small (fp16) turns the audio into text. Long audio is cut into 30 second chunks and the chunks are processed in batches.
- Every audio file is first converted to 16 kHz mono wav with ffmpeg, and the transcripts are saved as UTF-8.
- The transcript is passed as the prompt to an Unsloth dynamic 4-bit DeepSeek-R1-Distill-Qwen-7B model, which writes its thinking and a final answer.
- Whisper is removed from the GPU before the large model is loaded, and the reasoning questions are generated in one batch.